# IBM Data Analyst Capstone Project
## Module 3 — Data Wrangling

**Goal:** Clean the Stack Overflow Developer Survey dataset by:  
1. Finding and removing duplicate rows  
2. Identifying missing values  
3. Imputing (filling) missing values with sensible defaults  
4. Normalising compensation data  

The cleaned dataset is saved and used in all later modules.

In [1]:
# ── CELL 1: Import libraries ───────────────────────────────────────────────────
import pandas as pd    # data manipulation
import numpy as np     # numerical operations
import os              # file paths

In [2]:
# ── CELL 2: Load the survey dataset ───────────────────────────────────────────
# IBM-hosted Stack Overflow 2019 Developer Survey (cleaned subset)
DATA_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-DA0321EN-SkillsNetwork/LargeData/m1_survey_data.csv"
)

df = pd.read_csv(DATA_URL)

# Basic shape and column overview
print("Shape:", df.shape)
df.head()

Shape: (11552, 85)


,Respondent,MainBranch,Hobbyist,OpenSourcer,OpenSource,Employment,Country,Student,EdLevel,UndergradMajor,...,WelcomeChange,SONewContent,Age,Gender,Trans,Sexuality,Ethnicity,Dependents,SurveyLength,SurveyEase
0,4,I am a developer by profession,No,Never,The quality of OSS and closed source software ...,Employed full-time,United States,No,"Bachelor’s degree (BA, BS, B.Eng., etc.)","Computer science, computer engineering, or sof...",...,Just as welcome now as I felt last year,Tech articles written by other developers;Indu...,22.0,Man,No,Straight / Heterosexual,White or of European descent,No,Appropriate in length,Easy
1,9,I am a developer by profession,Yes,Once a month or more often,The quality of OSS and closed source software ...,Employed full-time,New Zealand,No,Some college/university study without earning ...,"Computer science, computer engineering, or sof...",...,Just as welcome now as I felt last year,NaN,23.0,Man,No,Bisexual,White or of European descent,No,Appropriate in length,Neither easy nor difficult
2,13,I am a developer by profession,Yes,Less than once a month but more than once per ...,"OSS is, on average, of HIGHER quality than pro...",Employed full-time,United States,No,"Master’s degree (MA, MS, M.Eng., MBA, etc.)","Computer science, computer engineering, or sof...",...,Somewhat more welcome now than last year,Tech articles written by other developers;Cour...,28.0,Man,No,Straight / Heterosexual,White or of European descent,Yes,Appropriate in length,Easy
3,16,I am a developer by profession,Yes,Never,The quality of OSS and closed source software ...,Employed full-time,United Kingdom,No,"Master’s degree (MA, MS, M.Eng., MBA, etc.)",NaN,...,Just as welcome now as I felt last year,Tech articles written by other developers;Indu...,26.0,Man,No,Straight / Heterosexual,White or of European descent,No,Appropriate in length,Neither easy nor difficult
4,17,I am a developer by profession,Yes,Less than once a month but more than once per ...,The quality of OSS and closed source software ...,Employed full-time,Australia,No,"Bachelor’s degree (BA, BS, B.Eng., etc.)","Computer science, computer engineering, or sof...",...,Just as welcome now as I felt last year,Tech articles written by other developers;Indu...,29.0,Man,No,Straight / Heterosexual,Hispanic or Latino/Latina;Multiracial,No,Appropriate in length,Easy


In [3]:
# ── CELL 3: Find duplicate rows ───────────────────────────────────────────────
dup_count = df.duplicated().sum()   # count rows that are exact duplicates
print(f"Duplicate rows found: {dup_count}")

Duplicate rows found: 154


In [4]:
# ── CELL 4: Remove duplicate rows ─────────────────────────────────────────────
df = df.drop_duplicates()           # keep only the first occurrence of each duplicate
df = df.reset_index(drop=True)      # reset the integer index after dropping rows

# Verify duplicates are gone
print(f"Duplicates after removal: {df.duplicated().sum()}")
print(f"New shape: {df.shape}")

Duplicates after removal: 0
New shape: (11398, 85)


In [5]:
# ── CELL 5: Identify missing values ───────────────────────────────────────────
# Count nulls per column; show only columns that have at least one missing value
missing = df.isnull().sum()
missing_cols = missing[missing > 0].sort_values(ascending=False)

print(f"Columns with missing values ({len(missing_cols)} total):")
print(missing_cols.to_string())

Columns with missing values (73 total):
BlockchainIs              2610
CodeRevHrs                2426
BlockchainOrg             2322
MiscTechWorkedWith        2182
SONewContent              1965
SOHowMuchTime             1917
WebFrameDesireNextYear    1617
MiscTechDesireNextYear    1455
WebFrameWorkedWith        1393
SOPartFreq                1128
DatabaseDesireNextYear    1042
ConvertedComp              816
CompTotal                  809
UndergradMajor             737
Ethnicity                  675
PlatformDesireNextYear     544
Sexuality                  542
ScreenName                 507
MgrMoney                   497
MgrIdiot                   493
MgrWant                    493
DatabaseWorkedWith         453
LastInt                    413
PlatformWorkedWith         411
SOVisit1st                 325
SocialMedia                293
Age                        287
CompFreq                   206
PurchaseHow                196
WorkChallenge              164
EduOther                   164

In [6]:
# ── CELL 6: Impute WorkLoc — fill with the most common (mode) value ───────────
# Inspect the distribution of WorkLoc before imputation
print("WorkLoc value counts BEFORE imputation:")
print(df["WorkLoc"].value_counts())

# The mode (most frequent value) is the best neutral fill for categorical data
workloc_mode = df["WorkLoc"].mode()[0]   # [0] extracts the string from the Series
df["WorkLoc"].fillna(workloc_mode, inplace=True)

print(f"\nFilled missing WorkLoc with: '{workloc_mode}'")
print(f"Missing WorkLoc after imputation: {df['WorkLoc'].isnull().sum()}")

WorkLoc value counts BEFORE imputation:
WorkLoc
Office                                            6806
Home                                              3589
Other place, such as a coworking space or cafe     971
Name: count, dtype: int64

Filled missing WorkLoc with: 'Office'
Missing WorkLoc after imputation: 0


C:\Users\Hamza\AppData\Local\Temp\ipykernel_22336\2026340637.py:8: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["WorkLoc"].fillna(workloc_mode, inplace=True)


In [7]:
# ── CELL 7: Impute ConvertedComp (salary) — fill with median ─────────────────
# Median is preferred over mean for salary data because extreme outliers skew the mean
salary_median = df["ConvertedComp"].median()
df["ConvertedComp"].fillna(salary_median, inplace=True)

print(f"Filled missing ConvertedComp with median: ${salary_median:,.0f}")
print(f"Missing ConvertedComp after imputation: {df['ConvertedComp'].isnull().sum()}")

Filled missing ConvertedComp with median: $57,745
Missing ConvertedComp after imputation: 0


C:\Users\Hamza\AppData\Local\Temp\ipykernel_22336\272420236.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["ConvertedComp"].fillna(salary_median, inplace=True)


In [8]:
# ── CELL 8: Impute EdLevel — fill with mode ───────────────────────────────────
edlevel_mode = df["EdLevel"].mode()[0]
df["EdLevel"].fillna(edlevel_mode, inplace=True)

print(f"Filled missing EdLevel with: '{edlevel_mode}'")
print(f"Missing EdLevel after imputation: {df['EdLevel'].isnull().sum()}")

Filled missing EdLevel with: 'Bachelor’s degree (BA, BS, B.Eng., etc.)'
Missing EdLevel after imputation: 0


C:\Users\Hamza\AppData\Local\Temp\ipykernel_22336\2224723385.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["EdLevel"].fillna(edlevel_mode, inplace=True)


In [9]:
# ── CELL 9: Final missing-value check ─────────────────────────────────────────
# Confirm the key columns we imputed now have zero nulls
key_cols = ["WorkLoc", "ConvertedComp", "EdLevel"]
print("Remaining nulls in key columns:")
print(df[key_cols].isnull().sum())

Remaining nulls in key columns:
WorkLoc          0
ConvertedComp    0
EdLevel          0
dtype: int64


In [10]:
# ── CELL 10: Summary statistics of cleaned dataset ────────────────────────────
# Quick statistical overview for numeric columns
df.describe()

,Respondent,CompTotal,ConvertedComp,WorkWeekHrs,CodeRevHrs,Age
count,11398.000000,1.058900e+04,1.139800e+04,11276.000000,8972.000000,11111.000000
mean,12490.392437,7.570477e+05,1.263096e+05,42.064606,4.781071,30.778895
std,7235.461999,9.705598e+06,2.846750e+05,24.672741,4.567060,7.393686
min,4.000000,0.000000e+00,0.000000e+00,3.000000,0.000000,16.000000
25%,6264.250000,2.500000e+04,2.901600e+04,40.000000,2.000000,25.000000
50%,12484.000000,6.500000e+04,5.774500e+04,40.000000,4.000000,29.000000
75%,18784.750000,1.200000e+05,9.500000e+04,43.000000,5.000000,35.000000
max,25142.000000,7.000000e+08,2.000000e+06,1012.000000,99.000000,99.000000


In [11]:
# ── CELL 11: Save cleaned dataset ─────────────────────────────────────────────
out_dir  = os.path.join("..", "data")
os.makedirs(out_dir, exist_ok=True)

out_path = os.path.join(out_dir, "survey_data_cleaned.csv")
df.to_csv(out_path, index=False)
print(f"Cleaned dataset saved → {out_path}")
print(f"Final shape: {df.shape}")

Cleaned dataset saved → ..\data\survey_data_cleaned.csv
Final shape: (11398, 85)
